# Shreve Week 11 — PDEs and SDEs

**Week 11** — PDEs and SDEs

This notebook teaches **pdes and sdes** in the style of our graduate probability notebook: precise definitions from Shreve, then **verified with Python**.

## What you will learn

| Part | Topic | Shreve |
|------|-------|--------|
| 1 | **SDE formulation** | Ch. 6.1 |
| 2 | **Feynman-Kac formula** | Ch. 6.2 |
| 3 | **BSM PDE ↔ expectation** | Ch. 6.2 |
| 4 | **Euler-Maruyama simulation** | Ch. 6.1 |
| — | **Problem set** | Ch. 6 |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — stochastic calculus is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem set** at the end has **click-to-reveal solutions**.
4. Primary reference: **Shreve**, *Stochastic Calculus for Finance II* — see chapter pointers in each section.

**Shreve reference:** Ch. 6 — [Vol II PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)

Let's begin.


---
## Setup — run this first

We use NumPy for simulation, SciPy for exact distributions, and Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)


---
# Part 1 — Stochastic Differential Equations

General SDE: $dX_t = \mu(t,X_t)\, dt + \sigma(t,X_t)\, dW_t$.

**Strong solution:** pathwise uniqueness. **Weak solution:** distribution uniqueness.

**Shreve Ch. 6.1:** SDEs and existence.


In [ ]:
# Ornstein-Uhlenbeck: dX = -κX dt + σ dW
def simulate_ou(x0, kappa, sigma, T, n, seed=110):
    rng = np.random.default_rng(seed)
    dt = T/n
    X = x0
    path = [X]
    for _ in range(n):
        dW = rng.normal(0, np.sqrt(dt))
        X = X + (-kappa*X)*dt + sigma*dW
        path.append(X)
    return np.array(path)

path = simulate_ou(1.0, 2.0, 0.3, 5.0, 1000)
plt.plot(path)
plt.title("Ornstein-Uhlenbeck path")
plt.show()


---
# Part 2 — Feynman-Kac Formula

If $V(t,x) = E^Q[e^{-r(T-t)} g(X_T) \mid X_t=x]$ and SDE for $X$, then $V$ solves:

$$\frac{\partial V}{\partial t} + \mu\frac{\partial V}{\partial x} + \frac{1}{2}\sigma^2\frac{\partial^2 V}{\partial x^2} - rV = 0$$

**Shreve Ch. 6.2:** bridge between PDE and expectation.


In [ ]:
# Feynman-Kac: E[g(W_T)] solves heat equation with terminal g
# For g(x)=x^2: V(0,0) = E[W_T^2] = T
T = 2.0
rng = np.random.default_rng(111)
W_T_sq = rng.normal(0, np.sqrt(T), 100_000)**2
print(f"E[W_T²] MC = {W_T_sq.mean():.4f}, PDE terminal = T = {T}")


---
# Part 3 — BSM as Feynman-Kac

Call price = $E^Q[e^{-rT}(S_T-K)^+]$ solves BSM PDE with $V(T,S)=\max(S-K,0)$.

**Shreve Ch. 6.2:** BSM PDE from Feynman-Kac.


In [ ]:
# PDE finite difference (simple) vs MC
S0, K, T, r, sig = 100, 100, 1, 0.05, 0.2
# MC already done in week 6
rng = np.random.default_rng(112)
Z = rng.standard_normal(500_000)
S_T = S0*np.exp((r-0.5*sig**2)*T + sig*np.sqrt(T)*Z)
mc = np.exp(-r*T)*np.maximum(S_T-K,0).mean()
print(f"Feynman-Kac (MC) call price = {mc:.4f}")


---
# Part 4 — Euler-Maruyama Scheme

$X_{n+1} = X_n + \mu\Delta t + \sigma\sqrt{\Delta t}\, Z_n$.

Discretizes SDE for simulation and PDE via Monte Carlo.

**Shreve Ch. 6.1:** numerical SDE methods.


In [ ]:
def euler_maruyama(mu_fn, sig_fn, x0, T, n, seed=113):
    rng = np.random.default_rng(seed)
    dt = T/n
    X = x0
    for _ in range(n):
        dW = rng.normal(0, np.sqrt(dt))
        X = X + mu_fn(X)*dt + sig_fn(X)*dW
    return X

# GBM terminal
em = [euler_maruyama(lambda x: r*x, lambda x: sig*x, S0, T, 1000, seed=s)
      for s in range(10_000)]
print(f"EM E[S_T] = {np.mean(em):.2f}, theory = {S0*np.exp(r*T):.2f}")


**Your turn:** Discretization error of Euler-Maruyama for GBM as $\Delta t$ decreases.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. State Feynman-Kac formula linking PDE to expectation.

2. Write OU SDE and its stationary distribution.

3. Derive BSM PDE from Feynman-Kac for $V=E^Q[e^{-rT}(S_T-K)^+]$.

4. What is the order of strong error for Euler-Maruyama?


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** $V(t,x)=E[e^{-r(T-t)}g(X_T)|X_t=x]$ solves $V_t + \mu V_x + \frac{1}{2}\sigma^2 V_{xx} - rV=0$.

**2.** $dX=-\kappa X dt + \sigma dW$; stationary $N(0, \sigma^2/(2\kappa))$.

**3.** $X=S$: $\mu=rS$, $\sigma=\sigma S$ gives BSM PDE.

**4.** Strong order $O(\sqrt{\Delta t})$ typically for EM.

</details>


---
## Further reading

- **Shreve**, *Stochastic Calculus for Finance II*, Ch. 6 — primary text for this week.
- **Shreve**, *Stochastic Calculus for Finance I* — discrete-time foundations (Ch. 1–5).
- **Karatzas & Shreve**, *Brownian Motion and Stochastic Calculus* — rigorous continuous-time theory.

Whenever a theorem says a process "converges" or a formula "holds in expectation," you can **simulate it** here and see the numbers match.
